# Очистка и преобразование данных в Python

## Единый учебный notebook для очного практического занятия

Этот notebook собран так, чтобы всё было в одном месте:

1. Генерация учебных данных.
2. Проверка окружения.
3. Короткое повторение Python.
4. Загрузка данных.
5. Диагностика качества.
6. Очистка данных.
7. Преобразование данных.
8. Объединение таблиц.
9. Группировки и сводные таблицы.
10. Сохранение результата.
11. Самостоятельная мини-практика.
12. Подсказки по типовым ошибкам.

## Как работать с notebook

Запускайте ячейки строго сверху вниз.

Если появилась ошибка:

1. Не переходите дальше.
2. Прочитайте последнюю строку ошибки.
3. Вернитесь к предыдущей ячейке.
4. Проверьте, создана ли нужная переменная.
5. Проверьте, правильно ли написано имя файла или столбца.

# Рабочая ситуация

Вы работаете аналитиком в компании, которая продаёт товары через разные каналы:

- online;
- marketplace;
- offline;
- partner.

Руководителю нужен подготовленный файл с продажами для отчёта.

В исходных данных специально заложены учебные ошибки:

- пропуски;
- дубликаты;
- некорректные даты;
- лишние пробелы;
- разные варианты написания одного значения;
- отрицательные количества;
- нулевые или отрицательные цены;
- некорректные скидки;
- несовпадения между основной таблицей и справочниками.

Ваша задача — подготовить данные так, чтобы по ним можно было считать выручку, строить группировки и делать выводы.

# Что получится в результате

В конце notebook будут созданы файлы в папке `outputs`:

| Файл | Назначение |
|---|---|
| `clean_sales_report.csv` | очищенная и обогащённая таблица продаж |
| `region_summary.csv` | показатели по регионам |
| `segment_summary.csv` | показатели по клиентским сегментам |
| `category_channel_pivot.xlsx` | сводная таблица по категориям и каналам |
| `additional_analysis.xlsx` | самостоятельный анализ |

Также вы сформулируете краткий аналитический вывод.

# Часть 1. Проверка окружения

Сначала проверим, что Python работает и мы понимаем, в какой папке выполняется notebook.

Это важно: если Python ищет файл не в той папке, появится ошибка `FileNotFoundError`.

In [ ]:
from pathlib import Path
import sys

print("Версия Python:")
print(sys.version)

print("\nТекущая рабочая папка:")
print(Path.cwd())

## Подсказка

`Path.cwd()` показывает текущую рабочую папку.

Если дальше Python не сможет найти файл, первое, что нужно проверить, — именно текущую папку.

# Часть 2. Импорт библиотек

Нам понадобятся:

- `pandas` — работа с таблицами;
- `numpy` — работа с числами и пропусками;
- `matplotlib` — базовая визуализация;
- `json` — запись JSON-файла;
- `random` — генерация учебных данных.

In [ ]:
try:
    import json
    import random
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    print("Библиотеки импортированы успешно")
    print("pandas:", pd.__version__)
    print("numpy:", np.__version__)
except ImportError as error:
    print("Ошибка импорта библиотеки:")
    print(error)

## Что делать, если появилась ошибка `ModuleNotFoundError`

Такой текст означает, что библиотека не установлена.

Варианты решения:

1. В Google Colab большинство нужных библиотек уже есть.
2. Если работаете локально, можно установить зависимости командой:

```python
%pip install pandas numpy matplotlib openpyxl
```

3. После установки перезапустите ячейку импорта.

# Часть 3. Создание учебных данных внутри notebook

Чтобы не зависеть от внешних файлов, этот notebook сам создаёт учебные данные.

Будут созданы папки:

```text
data/raw/
outputs/
```

И файлы:

```text
data/raw/sales.csv
data/raw/products.xlsx
data/raw/clients.csv
data/raw/regions.json
```

В данные специально добавлены ошибки, чтобы их можно было найти и исправить.

In [ ]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Папка исходных данных:", RAW_DIR)
print("Папка результатов:", OUTPUT_DIR)

## Создаём справочник товаров

Справочник товаров содержит:

- `product_id` — код товара;
- `product_name` — название товара;
- `category` — категория;
- `cost` — себестоимость.

В справочнике специально есть лишние пробелы и разный регистр в категориях.

In [ ]:
products = pd.DataFrame(
    [
        ["P001", "Laptop Basic 14", " Electronics", 42000],
        ["P002", "Laptop Pro 15", "electronics ", 68000],
        ["P003", "Wireless Mouse", "Accessories", 700],
        [" P004", "Mechanical Keyboard", " accessories", 2800],
        ["P005", "Office Chair", "Furniture", 6500],
        ["P006", "Standing Desk", "furniture ", 14500],
        ["P007", "USB-C Hub", "Accessories", 1800],
        ["P008", "Monitor 24", "Electronics", 11800],
        ["P009", "Monitor 27", " electronics", 16500],
        ["P010", "Webcam HD", "Accessories ", 2300],
        ["P011", "Notebook Stand", "accessories", 1200],
        ["P012", "Desk Lamp", "Furniture", 1900],
    ],
    columns=["product_id", "product_name", "category", "cost"],
)

display(products)

## Создаём справочник клиентов

Справочник клиентов содержит:

- `client_id` — код клиента;
- `segment` — сегмент клиента;
- `registration_date` — дата регистрации;
- `loyalty_level` — уровень лояльности.

В одном ключе клиента специально есть лишний пробел.

In [ ]:
segments = ["B2C", "B2B", "SMB", "Enterprise"]
loyalty_levels = ["Bronze", "Silver", "Gold", "Platinum"]

clients_data = []
for i in range(1, 31):
    clients_data.append(
        {
            "client_id": f"C{i:03d}",
            "segment": random.choice(segments),
            "registration_date": (
                pd.Timestamp("2025-01-01")
                + pd.Timedelta(days=random.randint(0, 420))
            ).strftime("%Y-%m-%d"),
            "loyalty_level": random.choice(loyalty_levels),
        }
    )

clients = pd.DataFrame(clients_data)

# Специальная учебная ошибка: лишний пробел в ключе клиента
clients.loc[5, "client_id"] = " C006"

display(clients.head(10))

## Создаём справочник регионов

Справочник регионов содержит:

- `region_id` — код региона;
- `region_name` — название региона;
- `macro_region` — макрорегион.

In [ ]:
regions = pd.DataFrame(
    [
        {"region_id": "R01", "region_name": "Москва", "macro_region": "Центральный"},
        {"region_id": "R02", "region_name": "Санкт-Петербург", "macro_region": "Северо-Запад"},
        {"region_id": "R03", "region_name": "Казань", "macro_region": "Приволжский"},
        {"region_id": "R04", "region_name": "Екатеринбург", "macro_region": "Уральский"},
        {"region_id": "R05", "region_name": "Новосибирск", "macro_region": "Сибирский"},
        {"region_id": "R06", "region_name": "Ростов-на-Дону", "macro_region": "Южный"},
        {"region_id": "R07", "region_name": "Владивосток", "macro_region": "Дальневосточный"},
    ]
)

display(regions)

## Создаём таблицу продаж

Основная таблица продаж содержит:

- `order_id` — номер заказа;
- `order_date` — дата заказа;
- `client_id` — код клиента;
- `product_id` — код товара;
- `region_id` — код региона;
- `channel` — канал продаж;
- `quantity` — количество;
- `unit_price` — цена за единицу;
- `discount` — скидка.

В эту таблицу специально добавлены ошибки качества.

In [ ]:
clean_product_ids = [value.strip() for value in products["product_id"].tolist()]
clean_client_ids = [value.strip() for value in clients["client_id"].tolist()]
region_ids = regions["region_id"].tolist()

channels = [
    "online",
    "Online",
    "ONLINE ",
    " marketplace",
    "marketplace",
    "offline",
    "Offline ",
    "partner",
    "PARTNER ",
]

orders = []

for i in range(1, 121):
    product_id = random.choice(clean_product_ids)
    client_id = random.choice(clean_client_ids)
    region_id = random.choice(region_ids)
    order_date = pd.Timestamp("2026-04-01") + pd.Timedelta(days=random.randint(0, 75))

    quantity = random.randint(1, 7)
    unit_price = round(random.uniform(800, 85000), 2)
    discount = random.choice([0, 0.05, 0.10, 0.15, 0.20])

    orders.append(
        {
            "order_id": f"ORD-{1000 + i}",
            "order_date": order_date.strftime("%Y-%m-%d"),
            "client_id": client_id,
            "product_id": product_id,
            "region_id": region_id,
            "channel": random.choice(channels),
            "quantity": quantity,
            "unit_price": unit_price,
            "discount": discount,
        }
    )

sales = pd.DataFrame(orders)

# Добавляем контролируемые учебные ошибки

# 1. Дубликат заказа
duplicate_row = sales.loc[9].copy()
sales = pd.concat([sales, pd.DataFrame([duplicate_row])], ignore_index=True)

# 2. Пропуски региона
sales.loc[3, "region_id"] = np.nan
sales.loc[24, "region_id"] = np.nan

# 3. Пропуски скидки
sales.loc[7, "discount"] = np.nan
sales.loc[39, "discount"] = np.nan

# 4. Некорректные даты
sales.loc[14, "order_date"] = "not_a_date"
sales.loc[52, "order_date"] = "2026/99/99"

# 5. Некорректные количества
sales.loc[18, "quantity"] = -2
sales.loc[43, "quantity"] = 0

# 6. Некорректные цены
sales.loc[21, "unit_price"] = 0
sales.loc[61, "unit_price"] = -1500

# 7. Некорректные скидки
sales.loc[28, "discount"] = 1.5
sales.loc[66, "discount"] = -0.1

# 8. Несуществующий товар
sales.loc[34, "product_id"] = "P999"

# 9. Несуществующий регион
sales.loc[46, "region_id"] = "R99"

# 10. Лишние пробелы в ключах
sales.loc[58, "product_id"] = " P003 "
sales.loc[75, "client_id"] = " C010 "

display(sales.head(15))
print("Размер sales:", sales.shape)

## Сохраняем учебные данные в файлы

Теперь сохраним созданные таблицы в разные форматы:

- продажи — CSV;
- товары — Excel;
- клиенты — CSV;
- регионы — JSON.

Так мы отработаем загрузку данных из разных источников.

In [ ]:
sales.to_csv(RAW_DIR / "sales.csv", index=False, encoding="utf-8")
products.to_excel(RAW_DIR / "products.xlsx", index=False)
clients.to_csv(RAW_DIR / "clients.csv", index=False, encoding="utf-8")

with open(RAW_DIR / "regions.json", "w", encoding="utf-8") as file:
    json.dump(
        regions.to_dict(orient="records"),
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Файлы созданы:")
print(RAW_DIR / "sales.csv")
print(RAW_DIR / "products.xlsx")
print(RAW_DIR / "clients.csv")
print(RAW_DIR / "regions.json")

# Часть 4. Проверяем наличие файлов

Даже если файлы созданы только что, полезно выполнить проверку.

Это формирует рабочую привычку аналитика: сначала проверить входные данные, потом анализировать.

In [ ]:
required_files = [
    Path("data/raw/sales.csv"),
    Path("data/raw/products.xlsx"),
    Path("data/raw/clients.csv"),
    Path("data/raw/regions.json"),
]

print("Проверяем файлы:")

for file_path in required_files:
    if file_path.exists():
        print("OK:", file_path)
    else:
        print("НЕ НАЙДЕН:", file_path)

# Часть 5. Мини-повторение Python

Перед pandas коротко вспомним базовые элементы Python.

Эта часть нужна, чтобы код дальше был понятнее.

## Переменные

Переменная — это имя, под которым хранится значение.

Пример: мы можем сохранить количество, цену и скидку, а затем посчитать выручку.

In [ ]:
quantity = 3
unit_price = 1250
discount = 0.10

revenue = quantity * unit_price * (1 - discount)

print("Количество:", quantity)
print("Цена:", unit_price)
print("Скидка:", discount)
print("Выручка:", revenue)

## Список

Список хранит несколько значений.

Например, список каналов продаж.

In [ ]:
channels_list = ["online", "marketplace", "offline", "partner"]

print(channels_list)
print("Первый элемент:", channels_list[0])
print("Количество элементов:", len(channels_list))

## Словарь

Словарь хранит значения по ключам.

Это похоже на одну строку таблицы: есть название поля и значение.

In [ ]:
one_order = {
    "order_id": "ORD-1001",
    "channel": "online",
    "quantity": 3,
    "unit_price": 1250,
}

print(one_order)
print("Канал продаж:", one_order["channel"])

## Условие

Условие помогает проверить правило.

Например: количество должно быть больше нуля.

In [ ]:
quantity = -2

if quantity <= 0:
    print("Ошибка: количество должно быть больше нуля")
else:
    print("Количество корректное")

## Цикл

Цикл повторяет одно действие несколько раз.

Например, можно проверить список файлов.

In [ ]:
file_names = ["sales.csv", "products.xlsx", "clients.csv", "regions.json"]

for file_name in file_names:
    print("Нужно проверить файл:", file_name)

## Функция

Функция нужна, чтобы повторять действие по понятному имени.

In [ ]:
def calculate_revenue(quantity, unit_price, discount):
    result = quantity * unit_price * (1 - discount)
    return result

print(calculate_revenue(2, 1000, 0.10))
print(calculate_revenue(5, 500, 0))

# Часть 6. Загружаем данные из файлов

Теперь загрузим файлы, которые были созданы выше.

Важно: дальше работаем не с таблицами, которые лежат в памяти после генерации, а именно с файлами. Это ближе к реальной работе аналитика.

In [ ]:
sales = pd.read_csv("data/raw/sales.csv")
products = pd.read_excel("data/raw/products.xlsx")
clients = pd.read_csv("data/raw/clients.csv")
regions = pd.read_json("data/raw/regions.json")

print("Данные загружены")
print("sales:", sales.shape)
print("products:", products.shape)
print("clients:", clients.shape)
print("regions:", regions.shape)

## Смотрим первые строки

`head()` показывает первые строки таблицы.

Это нужно, чтобы быстро понять структуру данных.

In [ ]:
display(sales.head())

In [ ]:
display(products.head())

In [ ]:
display(clients.head())

In [ ]:
display(regions.head())

# Часть 7. Первичная диагностика таблицы продаж

Перед очисткой нужно понять, что именно не так с данными.

Проверяем:

1. Размер таблицы.
2. Названия столбцов.
3. Типы данных.
4. Пропуски.
5. Дубликаты.
6. Некорректные значения.

In [ ]:
print("Размер sales:")
print(sales.shape)

print("\nСтолбцы sales:")
print(sales.columns.tolist())

## Проверяем типы данных

Типы данных показывают, как pandas понял каждый столбец.

Если дата прочиталась как `object`, значит пока это текст, а не дата.

In [ ]:
display(sales.dtypes)
sales.info()

## Проверяем пропуски

`isna().sum()` показывает количество пропусков по каждому столбцу.

In [ ]:
display(sales.isna().sum())

## Проверяем дубликаты

Заказ должен быть уникальным по `order_id`.

Если один и тот же `order_id` встречается повторно, показатели могут быть завышены.

In [ ]:
duplicates_count = sales["order_id"].duplicated().sum()
print("Количество дубликатов order_id:", duplicates_count)

duplicate_rows = sales[sales["order_id"].duplicated(keep=False)]
display(duplicate_rows)

## Проверяем текстовые значения

Для человека `online`, `Online` и `ONLINE ` выглядят похоже.

Для Python это разные значения.

Поэтому перед анализом нужно нормализовать текст.

In [ ]:
print("Уникальные значения channel:")
display(sales["channel"].unique())

## Проверяем числовые бизнес-правила

Правила:

| Поле | Корректное значение |
|---|---|
| `quantity` | больше 0 |
| `unit_price` | больше 0 |
| `discount` | от 0 до 1 |

In [ ]:
print("Минимальное quantity:", sales["quantity"].min())
print("Минимальное unit_price:", sales["unit_price"].min())
print("Минимальное discount:", sales["discount"].min())
print("Максимальное discount:", sales["discount"].max())

In [ ]:
print("Строки с quantity <= 0")
display(sales[sales["quantity"] <= 0])

print("Строки с unit_price <= 0")
display(sales[sales["unit_price"] <= 0])

print("Строки со скидкой вне диапазона 0-1")
display(
    sales[
        (sales["discount"] < 0) |
        (sales["discount"] > 1)
    ]
)

## Карта проблем данных

Заполните список своими словами.

Пример:

1. В `region_id` есть пропуски.
2. В `discount` есть пропуски.
3. В `order_id` есть дубликаты.
4. В `channel` разные варианты написания.
5. В `quantity` есть значения меньше или равные нулю.
6. В `unit_price` есть значения меньше или равные нулю.
7. В `discount` есть значения вне диапазона от 0 до 1.
8. В `order_date` есть некорректные даты.

### Моя карта проблем

1. ...
2. ...
3. ...
4. ...
5. ...
6. ...
7. ...
8. ...

# Часть 8. Очистка данных

Перед очисткой создадим копию таблицы.

Так исходная таблица останется без изменений.

In [ ]:
sales_clean = sales.copy()

print("Исходная таблица:", sales.shape)
print("Рабочая копия:", sales_clean.shape)

## Очищаем `channel`

Используем:

- `.str.strip()` — убрать пробелы в начале и конце;
- `.str.lower()` — привести к нижнему регистру.

In [ ]:
print("До очистки:")
display(sales_clean["channel"].unique())

sales_clean["channel"] = sales_clean["channel"].str.strip().str.lower()

print("После очистки:")
display(sales_clean["channel"].unique())

## Преобразуем дату заказа

`pd.to_datetime()` преобразует значения в дату.

`errors="coerce"` означает: если значение нельзя преобразовать, поставить пропуск.

In [ ]:
print("Тип order_date до преобразования:")
print(sales_clean["order_date"].dtype)

sales_clean["order_date"] = pd.to_datetime(
    sales_clean["order_date"],
    errors="coerce"
)

print("Тип order_date после преобразования:")
print(sales_clean["order_date"].dtype)

print("Количество некорректных дат:")
print(sales_clean["order_date"].isna().sum())

display(sales_clean[sales_clean["order_date"].isna()])

## Удаляем строки с некорректными датами

В учебной работе удалим строки, где дата не преобразовалась.

В реальной задаче такие строки лучше уточнять у владельца данных.

In [ ]:
print("Строк до удаления некорректных дат:", len(sales_clean))

sales_clean = sales_clean.dropna(subset=["order_date"])

print("Строк после удаления некорректных дат:", len(sales_clean))

## Удаляем дубликаты заказов

Дубликаты удаляем по `order_id`.

In [ ]:
print("Дубликатов до очистки:", sales_clean["order_id"].duplicated().sum())
print("Строк до удаления дубликатов:", len(sales_clean))

sales_clean = sales_clean.drop_duplicates(subset=["order_id"])

print("Строк после удаления дубликатов:", len(sales_clean))
print("Дубликатов после очистки:", sales_clean["order_id"].duplicated().sum())

## Обрабатываем пропуски

Правила для учебной работы:

| Столбец | Что делаем | Почему |
|---|---|---|
| `region_id` | заменяем на `UNKNOWN` | регион неизвестен, но строку сохраняем |
| `discount` | заменяем на `0` | если скидка не указана, считаем, что скидки нет |

In [ ]:
print("Пропуски до обработки:")
display(sales_clean.isna().sum())

sales_clean["region_id"] = sales_clean["region_id"].fillna("UNKNOWN")
sales_clean["discount"] = sales_clean["discount"].fillna(0)

print("Пропуски после обработки:")
display(sales_clean.isna().sum())

## Удаляем некорректные количества

`quantity` должно быть больше 0.

In [ ]:
invalid_quantity = sales_clean[sales_clean["quantity"] <= 0]

display(invalid_quantity)
print("Количество проблемных строк:", len(invalid_quantity))

print("Строк до фильтрации:", len(sales_clean))
sales_clean = sales_clean[sales_clean["quantity"] > 0]
print("Строк после фильтрации:", len(sales_clean))
print("Минимальное quantity:", sales_clean["quantity"].min())

## Удаляем некорректные цены

`unit_price` должна быть больше 0.

In [ ]:
invalid_price = sales_clean[sales_clean["unit_price"] <= 0]

display(invalid_price)
print("Количество проблемных строк:", len(invalid_price))

print("Строк до фильтрации:", len(sales_clean))
sales_clean = sales_clean[sales_clean["unit_price"] > 0]
print("Строк после фильтрации:", len(sales_clean))
print("Минимальное unit_price:", sales_clean["unit_price"].min())

## Проверяем скидки

`discount` должна быть от 0 до 1.

Примеры:

- `0` — скидки нет;
- `0.10` — скидка 10%;
- `0.25` — скидка 25%;
- `1` — скидка 100%.

In [ ]:
invalid_discount = sales_clean[
    (sales_clean["discount"] < 0) |
    (sales_clean["discount"] > 1)
]

display(invalid_discount)
print("Количество проблемных строк:", len(invalid_discount))

print("Строк до фильтрации:", len(sales_clean))
sales_clean = sales_clean[
    (sales_clean["discount"] >= 0) &
    (sales_clean["discount"] <= 1)
]
print("Строк после фильтрации:", len(sales_clean))
print("Минимальная скидка:", sales_clean["discount"].min())
print("Максимальная скидка:", sales_clean["discount"].max())

## Итоговая проверка после очистки

Проверим, что основные проблемы исправлены.

In [ ]:
print("Размер после очистки:", sales_clean.shape)

print("\nПропуски:")
display(sales_clean.isna().sum())

print("\nДубликаты order_id:")
print(sales_clean["order_id"].duplicated().sum())

print("\nМинимальное quantity:")
print(sales_clean["quantity"].min())

print("\nМинимальное unit_price:")
print(sales_clean["unit_price"].min())

print("\nДиапазон discount:")
print(sales_clean["discount"].min(), "-", sales_clean["discount"].max())

# Часть 9. Преобразование данных

Теперь создадим новые аналитические признаки.

## Рассчитываем выручку

Формула:

```text
revenue = quantity * unit_price * (1 - discount)
```

In [ ]:
sales_clean["revenue"] = (
    sales_clean["quantity"] *
    sales_clean["unit_price"] *
    (1 - sales_clean["discount"])
)

display(
    sales_clean[
        ["quantity", "unit_price", "discount", "revenue"]
    ].head()
)

print("Минимальная выручка:", sales_clean["revenue"].min())

## Создаём месяц заказа

Это нужно, чтобы анализировать продажи по месяцам.

In [ ]:
sales_clean["order_month"] = sales_clean["order_date"].dt.to_period("M").astype(str)

display(
    sales_clean[
        ["order_date", "order_month"]
    ].head()
)

# Часть 10. Подготовка справочников и объединение таблиц

Перед объединением нужно очистить ключи:

- `product_id`;
- `region_id`;
- `client_id`.

Частая ошибка: ключи выглядят одинаково для человека, но отличаются пробелами.

## Готовим справочник товаров

In [ ]:
print("До очистки products:")
display(products.head())

products["product_id"] = products["product_id"].str.strip()
products["product_name"] = products["product_name"].str.strip()
products["category"] = products["category"].str.strip().str.lower()

print("После очистки products:")
display(products.head())

## Очищаем `product_id` в продажах и объединяем с товарами

In [ ]:
sales_clean["product_id"] = sales_clean["product_id"].str.strip()

sales_products = sales_clean.merge(
    products,
    on="product_id",
    how="left"
)

print("Строк в sales_clean:", len(sales_clean))
print("Строк в sales_products:", len(sales_products))

display(sales_products.head())

## Проверяем, какие товары не нашлись

Если после объединения в `product_name` есть пропуски, значит `product_id` не найден в справочнике товаров.

In [ ]:
missing_products = sales_products["product_name"].isna().sum()

print("Количество строк без найденного товара:", missing_products)

display(
    sales_products[
        sales_products["product_name"].isna()
    ]
)

In [ ]:
sales_products["product_name"] = sales_products["product_name"].fillna("unknown_product")
sales_products["category"] = sales_products["category"].fillna("unknown_category")
sales_products["cost"] = sales_products["cost"].fillna(0)

print("Пропуски после обработки товаров:")
display(sales_products[["product_name", "category", "cost"]].isna().sum())

## Готовим регионы и объединяем

In [ ]:
regions["region_id"] = regions["region_id"].str.strip()
sales_products["region_id"] = sales_products["region_id"].str.strip()

sales_full = sales_products.merge(
    regions,
    on="region_id",
    how="left"
)

print("Строк до объединения:", len(sales_products))
print("Строк после объединения:", len(sales_full))

display(sales_full.head())

In [ ]:
missing_regions = sales_full["region_name"].isna().sum()

print("Количество строк без найденного региона:", missing_regions)

display(
    sales_full[
        sales_full["region_name"].isna()
    ]
)

sales_full["region_name"] = sales_full["region_name"].fillna("unknown_region")
sales_full["macro_region"] = sales_full["macro_region"].fillna("unknown_macro_region")

## Готовим клиентов и объединяем

In [ ]:
clients["client_id"] = clients["client_id"].str.strip()
clients["segment"] = clients["segment"].str.strip()
clients["loyalty_level"] = clients["loyalty_level"].str.strip()

sales_full["client_id"] = sales_full["client_id"].str.strip()

sales_full = sales_full.merge(
    clients,
    on="client_id",
    how="left"
)

print("Размер итоговой таблицы:", sales_full.shape)
display(sales_full.head())

In [ ]:
print("Пропуски после объединения с клиентами:")
display(
    sales_full[
        ["segment", "registration_date", "loyalty_level"]
    ].isna().sum()
)

sales_full["segment"] = sales_full["segment"].fillna("unknown_segment")
sales_full["loyalty_level"] = sales_full["loyalty_level"].fillna("unknown_loyalty")

## Проверяем итоговую таблицу

В ней должны быть данные заказа, товара, региона, клиента и рассчитанная выручка.

In [ ]:
print("Размер итоговой таблицы:")
print(sales_full.shape)

print("\nСтолбцы итоговой таблицы:")
print(sales_full.columns.tolist())

display(sales_full.head())
sales_full.info()

# Часть 11. Аналитические таблицы

Теперь данные подготовлены. Можно считать показатели.

## Показатели по регионам

Считаем:

- количество заказов;
- суммарную выручку;
- среднюю выручку заказа.

In [ ]:
region_summary = (
    sales_full
    .groupby("region_name", as_index=False)
    .agg(
        orders_count=("order_id", "nunique"),
        total_revenue=("revenue", "sum"),
        avg_order_revenue=("revenue", "mean")
    )
    .sort_values("total_revenue", ascending=False)
)

display(region_summary)

## Показатели по категориям

In [ ]:
category_summary = (
    sales_full
    .groupby("category", as_index=False)
    .agg(
        orders_count=("order_id", "nunique"),
        total_quantity=("quantity", "sum"),
        total_revenue=("revenue", "sum"),
        avg_order_revenue=("revenue", "mean")
    )
    .sort_values("total_revenue", ascending=False)
)

display(category_summary)

## Сводная таблица по категориям и каналам

Строки — категории.  
Столбцы — каналы.  
Значения — сумма выручки.

In [ ]:
category_channel_pivot = pd.pivot_table(
    sales_full,
    index="category",
    columns="channel",
    values="revenue",
    aggfunc="sum",
    fill_value=0
)

display(category_channel_pivot)

## Показатели по месяцам

In [ ]:
monthly_summary = (
    sales_full
    .groupby("order_month", as_index=False)
    .agg(
        orders_count=("order_id", "nunique"),
        total_revenue=("revenue", "sum"),
        avg_order_revenue=("revenue", "mean")
    )
    .sort_values("order_month")
)

display(monthly_summary)

## Показатели по клиентским сегментам

In [ ]:
segment_summary = (
    sales_full
    .groupby("segment", as_index=False)
    .agg(
        orders_count=("order_id", "nunique"),
        total_revenue=("revenue", "sum"),
        avg_order_revenue=("revenue", "mean")
    )
    .sort_values("total_revenue", ascending=False)
)

display(segment_summary)

# Часть 12. Простая визуализация

График помогает быстро увидеть различия между группами.

Построим столбчатую диаграмму выручки по регионам.

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(region_summary["region_name"], region_summary["total_revenue"])
plt.title("Выручка по регионам")
plt.xlabel("Регион")
plt.ylabel("Выручка")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Вопрос для самопроверки

По графику ответьте:

1. В каком регионе выручка самая высокая?
2. Есть ли регионы с заметно меньшей выручкой?
3. Можно ли делать окончательный управленческий вывод только по этому графику?

Подсказка: окончательный вывод требует проверки качества данных и понимания контекста.

# Часть 13. Самостоятельная мини-практика

Используйте таблицу `sales_full`.

Нужно получить три результата:

1. Топ-3 региона по выручке.
2. Топ-3 категории по количеству заказов.
3. Средняя выручка заказа по каналам продаж.

Подсказка: используйте `groupby`, `agg`, `sort_values`, `head`.

## Задание 1. Топ-3 региона по выручке

In [ ]:
top_regions = (
    sales_full
    .groupby("region_name", as_index=False)
    .agg(
        total_revenue=("revenue", "sum")
    )
    .sort_values("total_revenue", ascending=False)
    .head(3)
)

display(top_regions)

## Задание 2. Топ-3 категории по количеству заказов

In [ ]:
top_categories_by_orders = (
    sales_full
    .groupby("category", as_index=False)
    .agg(
        orders_count=("order_id", "nunique")
    )
    .sort_values("orders_count", ascending=False)
    .head(3)
)

display(top_categories_by_orders)

## Задание 3. Средняя выручка заказа по каналам

In [ ]:
channel_avg_order = (
    sales_full
    .groupby("channel", as_index=False)
    .agg(
        avg_order_revenue=("revenue", "mean"),
        orders_count=("order_id", "nunique")
    )
    .sort_values("avg_order_revenue", ascending=False)
)

display(channel_avg_order)

# Часть 14. Сохраняем результаты

Теперь сохраним итоговые файлы в папку `outputs`.

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

sales_full.to_csv(output_dir / "clean_sales_report.csv", index=False)
region_summary.to_csv(output_dir / "region_summary.csv", index=False)
segment_summary.to_csv(output_dir / "segment_summary.csv", index=False)
category_channel_pivot.to_excel(output_dir / "category_channel_pivot.xlsx")

with pd.ExcelWriter(output_dir / "additional_analysis.xlsx") as writer:
    top_regions.to_excel(writer, sheet_name="top_regions", index=False)
    top_categories_by_orders.to_excel(writer, sheet_name="top_categories", index=False)
    channel_avg_order.to_excel(writer, sheet_name="channel_avg_order", index=False)

print("Файлы сохранены:")
for file_path in output_dir.iterdir():
    print(file_path)

# Часть 15. Аналитический вывод

Заполните вывод по шаблону.

Не пишите слишком общий вывод вроде:

> Данные были проанализированы.

Хороший вывод должен ссылаться на конкретные показатели.

## Мой аналитический вывод

1. Больше всего выручки получено в регионе: ...
2. Самая сильная категория по выручке: ...
3. Наиболее заметный канал продаж: ...
4. Перед использованием результата нужно учитывать, что часть данных была очищена или исключена по правилам качества.
5. Следующий шаг анализа может быть таким: ...

# Часть 16. Финальная самопроверка

| Проверка | Статус |
|---|---|
| Notebook запускается сверху вниз |  |
| Учебные данные созданы |  |
| Файлы в `data/raw` существуют |  |
| Данные загружены из CSV, XLSX и JSON |  |
| Проверены размеры таблиц |  |
| Проверены первые строки |  |
| Проверены типы данных |  |
| Найдены пропуски |  |
| Найдены дубликаты |  |
| Очищен `channel` |  |
| Преобразован `order_date` |  |
| Удалены дубликаты `order_id` |  |
| Обработаны пропуски |  |
| Исключены некорректные количества |  |
| Исключены некорректные цены |  |
| Проверены скидки |  |
| Рассчитан `revenue` |  |
| Создан `order_month` |  |
| Выполнено объединение с товарами |  |
| Выполнено объединение с регионами |  |
| Выполнено объединение с клиентами |  |
| Построены группировки |  |
| Построена сводная таблица |  |
| Построен график |  |
| Итоговые файлы сохранены |  |
| Написан аналитический вывод |  |

# Часть 17. Типовые ошибки и простые исправления

| Ошибка | Что означает | Что проверить |
|---|---|---|
| `FileNotFoundError` | Python не нашёл файл | путь, папку, имя файла |
| `ModuleNotFoundError` | не установлена библиотека | импорт и установку пакетов |
| `KeyError` | нет такого столбца | `df.columns.tolist()` |
| `NameError` | переменная не создана | выполнены ли предыдущие ячейки |
| `TypeError` | операция не подходит для типа данных | типы через `df.dtypes` |
| `ValueError` | значение не подходит для операции | конкретное значение и формат |
| CSV открылся одной колонкой | неверный разделитель | попробовать `sep=";"` |
| Даты стали пропусками | есть плохие даты | посмотреть строки с `isna()` |
| После `merge` появились пропуски | ключ не найден в справочнике | проверить ключи и пробелы |
| После `merge` стало больше строк | дубликаты в справочнике | проверить уникальность ключей |

# Часть 18. Мини-шпаргалка pandas

## Посмотреть первые строки

```python
df.head()
```

## Посмотреть размер

```python
df.shape
```

## Посмотреть столбцы

```python
df.columns.tolist()
```

## Посмотреть типы

```python
df.dtypes
```

## Посмотреть пропуски

```python
df.isna().sum()
```

## Посмотреть уникальные значения

```python
df["column"].unique()
```

## Отфильтровать строки

```python
df[df["column"] > 0]
```

## Сгруппировать данные

```python
df.groupby("group_column").agg(total=("value_column", "sum"))
```

## Объединить таблицы

```python
left_table.merge(right_table, on="key", how="left")
```

## Сохранить CSV

```python
df.to_csv("result.csv", index=False)
```

# Конец практической работы

Главная идея занятия:

> Аналитик должен не только посчитать показатель, но и проверить, можно ли доверять данным, на которых этот показатель построен.

Вы прошли полный цикл:

1. Создали учебные данные.
2. Проверили файлы.
3. Загрузили таблицы.
4. Нашли ошибки.
5. Очистили данные.
6. Преобразовали признаки.
7. Объединили справочники.
8. Рассчитали показатели.
9. Построили сводную таблицу и график.
10. Сохранили результат.
11. Сформулировали вывод.